# Cube reorientation — algorithm bake-off

Closed-loop in-hand cube reorientation with the LEAP hand (16 actuators): rotate the
cube toward the goal quaternion while keeping it positioned in the palm. Same harness
and algorithms as the other env notebooks.

> **Note:** the cube face textures under `envs/cube/reorientation_cube_textures/` were
> missing from the repo; they're currently **placeholder solid colors** (one per face).
> Physics is unaffected — drop in the real decals any time to replace them.

In [1]:
# Always pick up the latest code in analytic_mppi/ (incl. experimental.py)
# without restarting the kernel — edit a .py, re-run a cell, done.
%load_ext autoreload
%autoreload 2

import numpy as np
from analytic_mppi.tasks import make_task
from analytic_mppi.dynamics import MujocoBackend
from analytic_mppi.eval import (
    Config, run_study, plot_study, summarize, render_video,
    cube_metrics, cube_panels,
)
from analytic_mppi.controllers import MPPIv2, MppiCma, CEM, DIAL, PredictiveSampling
from analytic_mppi.controllers.experimental import UniformGD, GaussianGD, RankCMA, FplValueCMA

TASK = "cube"
task = make_task(TASK)
DT = MujocoBackend(task.model_path).dt
print(f"task={TASK}  nq={task.nq} nv={task.nv} nu={task.nu} dt={DT}")

task=cube  nq=23 nv=22 nu=16 dt=0.01


## Experiment configuration

Each `Config` is one row: a label, the controller **class**, the cost mode, and its algo kwargs. Shared sampler settings apply to every row.

In [2]:
# cube is 16-DOF; keep K / steps modest in the template and scale up as needed
SHARED = dict(num_samples=128, plan_horizon=0.5, num_knots=4)

CONFIGS = [
    Config("mppi  (normal)",          MPPIv2,       "normal",         dict(noise_level=0.2, temperature=0.1)),
    Config("mppi_cma  (normal)",      MppiCma,      "normal",         dict(initial_noise_level=0.3, temperature=0.1, minimum_noise_level=0.1)),
    Config("mppi  (fpl)",             MPPIv2,       "fpl_discounted", dict(noise_level=0.2, temperature=0.1)),
    Config("uniform_gd  (fpl)",       UniformGD,    "fpl_discounted", dict(noise_level=0.3, n_gd_iter=5)),
    Config("gaussian_gd  (fpl)",      GaussianGD,   "fpl_discounted", dict(noise_level=0.2, n_gd_iter=5)),
    Config("rank_cma  (fpl)",         RankCMA,      "fpl_discounted", dict(sigma_init=0.3, n_gd_iter=5)),
    Config("fpl_value_cma  (fpl)",    FplValueCMA,  "fpl_discounted", dict(sigma_init=0.3, beta=10.0, n_gd_iter=5)),
]

## Run the closed-loop study

Fresh MuJoCo sim per (config, episode); mean ± 1 std bands across episodes.

In [3]:
STEPS = 200
N_EPISODES = 2

study = run_study(TASK, CONFIGS, steps=STEPS, n_episodes=N_EPISODES, **SHARED)

plot_study(study, lambda r: cube_metrics(r, task), cube_panels(), dt=DT,
           title=f"Cube reorientation — {N_EPISODES} ep × {STEPS} steps")
summarize(study, lambda r: cube_metrics(r, task), ["pos_err", "ori_err", "u_mag"])

running mppi  (normal)                          .. done
running mppi_cma  (normal)                      .

KeyboardInterrupt: 

## (optional) Render a video of one config

Set `RUN_VIDEO = True` to render an mp4 of a single config and embed it.

In [6]:
RUN_VIDEO = True
if RUN_VIDEO:
    from IPython.display import Video, display
    res = render_video(TASK, RankCMA, steps=10000, out_path="/tmp/cube_rank_cma.mp4",
                       cost_mode="fpl_discounted", sigma_init=0.3, n_gd_iter=5, **SHARED)
    print(f"plan {res['plan_ms']:.1f} ms/step")
    display(Video(str(res["path"]), embed=True, width=360))
else:
    print("set RUN_VIDEO = True to render")

plan 23.0 ms/step
